In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, when, trim, lower, count, avg, desc, lit

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schema_origen", "bronze")
dbutils.widgets.text("schema_destino", "silver")
dbutils.widgets.text("catalog", "catalog_pry_final")

In [0]:
schema_origen = dbutils.widgets.get("schema_origen")
schema_destino = dbutils.widgets.get("schema_destino")
catalog = dbutils.widgets.get("catalog")

In [0]:
fact_orders = spark.table(f"{catalog}.{schema_origen}.ordenes") \
    .withColumn("es_primera_orden", when(col("dias_desde_orden").isNull(), True).otherwise(False)) \
    .withColumn("dias_desde_orden", when(col("dias_desde_orden").isNull(), lit(0.0))
                                          .otherwise(col("dias_desde_orden"))) \
    .drop("ingestion_date")


fact_order_items = spark.table(f"{catalog}.{schema_origen}.ordenes_prior").drop("ingestion_date")

products_clean =spark.table(f"{catalog}.{schema_origen}.productos").withColumn("producto_nombre", trim(lower(col("producto_nombre"))))
aisles_clean = spark.table(f"{catalog}.{schema_origen}.pasillo").withColumn("pasillo", trim(lower(col("pasillo"))))
departments_clean = spark.table(f"{catalog}.{schema_origen}.departamentos").withColumn("departamento", trim(lower(col("departamento"))))

dim_products = products_clean.alias("p") \
    .join(aisles_clean.alias("a"), "pasillo_id", "left") \
    .join(departments_clean.alias("d"), "departamento_id", "left") \
    .select(
        col("p.producto_id"),
        col("p.producto_nombre"),
        col("a.pasillo_id"),
        col("a.pasillo").alias("pasillo_nombre"),
        col("d.departamento_id"),
        col("d.departamento").alias("departamento_nombre")
    )

dim_products.display()

In [0]:
dim_products.write.mode("overwrite").insertInto(f"{catalog}.{schema_destino}.dim_productos")